# 11 — Simulation-tuned alpha (CTL-03)

**Decision:** ADR 0060 / CTL-03=B — grid-search the demand fractile `alpha` on closed-loop **episode profit** (SIM-01=B), not the textbook newsvendor ratio `c_s / (c_s + c_w)`.

This notebook uses the production API in `sim/alpha_tune.py`:

- `evaluate_alpha_episode_profit` — score one (arm, alpha) under shared CRN
- `tune_alpha_grid` — pick best alpha on a grid per ladder arm
- `save_tuned_alpha_table` / `load_tuned_alpha_table` — artifact I/O

Scoring routes through the **Rust `voi_core` kernel** when `blueberries_voi._core.evaluate_alpha_tune_episode_py` is available (after `maturin develop`).

**Defaults are smoke-sized** — short horizon, CI alpha grid, cool shipment fixture. Set `FULL_RUN = True` only when you have time.


## Setup

From the repo root:

```bash
uv sync --extra notebooks --extra viz
uv run maturin develop --manifest-path crates/voi_py/Cargo.toml
uv run jupyter lab
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.sim.alpha_tune import (
    DEFAULT_CI_ALPHAS,
    DEFAULT_DESKTOP_ALPHAS,
    DEFAULT_TUNED_ALPHA_PATH,
    LADDER_ALPHA_ARMS,
    evaluate_alpha_episode_profit,
    load_tuned_alpha_table,
    save_tuned_alpha_table,
    tune_alpha_grid,
)
from blueberries_voi.sim.profit import DEFAULT_PROFIT_COSTS
from blueberries_voi.sim.shipments import default_shipments, smoke_cool_shipments

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

# --- notebook knobs (keep smoke defaults for interactive runs) ---
FULL_RUN = False
USE_ABDELLA = False  # needs Abdella parquet on disk
ROOT_SEED = 42

if FULL_RUN:
    ALPHAS = DEFAULT_DESKTOP_ALPHAS
    N_BURN, N_SCORE = 14, 28
else:
    ALPHAS = tuple(DEFAULT_CI_ALPHAS)
    N_BURN, N_SCORE = 2, 5

AVAILABLE_ARMS = ("constant", "rung0", "sw")
PLACEHOLDER_ARMS = tuple(a for a in LADDER_ALPHA_ARMS if a not in AVAILABLE_ARMS)

shipments = default_shipments() if USE_ABDELLA else smoke_cool_shipments()
costs = DEFAULT_PROFIT_COSTS
ARTIFACT = REPO_ROOT / "experiments" / "tuned_alpha_notebook.json"

rust_fn = getattr(rust_core, "evaluate_alpha_tune_episode_py", None) if rust_core else None
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"alpha grid ({len(ALPHAS)}): {ALPHAS}")
print(f"episode: n_burn={N_BURN}, n_score={N_SCORE}")
print(f"shipments: {'Abdella' if USE_ABDELLA else 'smoke cool fixture'}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})


## Textbook fractile vs simulation tuning (CTL-03)

The single-period newsvendor fractile uses scaffold costs from `DEFAULT_PROFIT_COSTS` (uncalibrated, ADR 0104). CTL-03=B instead maximizes closed-loop episode profit on the grid below.


In [ ]:
alpha_theory_penalty = costs.stockout_penalty / (costs.stockout_penalty + costs.waste_cost)
alpha_theory_margin = costs.unit_margin / (costs.unit_margin + costs.waste_cost)
print(f"Textbook (penalty / waste): {alpha_theory_penalty:.3f}")
print(f"Textbook (margin / waste):  {alpha_theory_margin:.3f}")


## Score one (arm, alpha)

Quick sanity check before running the full grid.


In [ ]:
DEMO_ARM, DEMO_ALPHA = "sw", 0.9
demo_profit = evaluate_alpha_episode_profit(
    DEMO_ARM,
    DEMO_ALPHA,
    ROOT_SEED,
    shipments=shipments,
    costs=costs,
    n_burn=N_BURN,
    n_score=N_SCORE,
)
print(f"{DEMO_ARM} alpha={DEMO_ALPHA}: scored profit = {demo_profit:.2f}")


## Profit curve for one arm (tqdm over alpha grid)

Shared `ROOT_SEED` gives common random numbers across alpha candidates on the same arm.


In [ ]:
CURVE_ARM = "sw"
profits: list[float] = []
for alpha in tqdm(ALPHAS, desc=f"{CURVE_ARM} alpha grid"):
    profits.append(
        evaluate_alpha_episode_profit(
            CURVE_ARM,
            float(alpha),
            ROOT_SEED,
            shipments=shipments,
            costs=costs,
            n_burn=N_BURN,
            n_score=N_SCORE,
        )
    )

best_idx = int(np.argmax(profits))
best_alpha = float(ALPHAS[best_idx])

fig, ax = plt.subplots()
ax.plot(ALPHAS, profits, "o-", color="#2563eb", label=f"{CURVE_ARM} episode profit")
ax.axvline(best_alpha, color="#16a34a", linestyle="--", label=f"tuned alpha* = {best_alpha:.2f}")
ax.axvline(alpha_theory_penalty, color="#dc2626", linestyle=":", label="theory (penalty)")
ax.axvline(alpha_theory_margin, color="#f97316", linestyle=":", label="theory (margin)")
ax.set_xlabel("alpha (NB demand fractile)")
ax.set_ylabel("Episode profit (scored days)")
ax.set_title("CTL-03 alpha grid search")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

list(zip(ALPHAS, [round(p, 2) for p in profits]))


## Tune all available ladder arms

`rollout` and `dp` remain placeholders (hand-filled in the artifact). tqdm wraps the arm loop.


In [ ]:
tuned: dict[str, float] = {}
for arm in tqdm(AVAILABLE_ARMS, desc="tune arms"):
    tuned[arm] = tune_alpha_grid(
        arm,
        alphas=ALPHAS,
        root_seed=ROOT_SEED,
        shipments=shipments,
        costs=costs,
        n_burn=N_BURN,
        n_score=N_SCORE,
    )

for arm in PLACEHOLDER_ARMS:
    tuned[arm] = 0.9  # placeholder until T-030 / T-031

tuned


## Save tuned alpha artifact

Writes JSON under `experiments/` (default production path is `experiments/tuned_alpha.json`). This notebook uses a notebook-specific filename.


In [ ]:
header = {
    "notebook": "11_simulation_alpha_tuning",
    "full_run": FULL_RUN,
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "alphas": list(ALPHAS),
    "root_seed": ROOT_SEED,
    "rust_kernel": bool(rust_available() and rust_fn is not None),
}
save_tuned_alpha_table(
    ARTIFACT,
    tuned,
    header=header,
)
loaded = load_tuned_alpha_table(ARTIFACT)
print(f"Saved {ARTIFACT}")
print(loaded)


## Takeaways

1. **CTL-03=B** picks alpha by simulation profit, not the textbook newsvendor fractile.
2. **`tune_alpha_grid`** runs per ladder arm under a shared `root_seed` (CRN across alpha candidates).
3. **Rust path** — when `_core` is built, scoring uses f-native `voi_core` physics; rebuild with `maturin develop` after Rust changes.
4. **Scale up** — set `FULL_RUN = True` (desktop alpha grid + longer episode) for production-style tuning; expect minutes, not seconds.
